# 02 — The attractor: corrected statement and its verification

**Addresses Blocker 2.** Proposition 1 as written claims $\nabla_\theta L_{PDE} = 0$ at
$F=0$. The branch-parameter half of the argument does not hold:

$$\nabla_{\theta_{\text{branch}}} L_{PDE}
= 2R\,\frac{\partial R}{\partial F}\,\underbrace{\frac{\partial F}{\partial \boldsymbol\beta}}_{=\;\boldsymbol\tau\;\neq\;0}\,
\frac{\partial \boldsymbol\beta}{\partial\theta}$$

There is no factor of $\boldsymbol\beta$ in this chain, so nothing forces it to vanish.

### What is actually true (the claim to prove instead)

1. $\nabla_{\theta_{\text{trunk}}} L_{PDE} = 0$ **exactly** — the trunks are frozen.
2. The **mass residual vanishes identically**, $R_1 \equiv 0$, and so does its gradient.
   Half the physics loss is structurally blind. *This is the hyperbolic-specific part*
   and it is what fails to hold for diffusion–reaction.
3. The surviving gradient comes only from $R_2$, and it points toward
   $\partial_x(\tfrac12 g h^2) = -gh\,\partial_x b$ — the **lake-at-rest steady-state
   manifold**, not the wave dynamics.
4. In the well-balanced case $h_0 - b = \text{const}$, $F = 0$ is an **exact global
   minimum** of $L_{PDE}$.

Experiments 1–3 below verify each clause. Experiment 4 fixes the IC shortcut.

> Prior art to cite: Rohrhofer, Posch, Gößnitzer & Geiger, *On the Role of Fixed Points of
> Dynamical Systems in Training PINNs*, TMLR 2023 (arXiv:2203.13648); De Ryck, Mishra &
> Molinaro, *wPINNs*, on hyperbolic conservation laws.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from deeponet_tf import FourBranchDeepONet, swe_residual, gradient_split_at_F0, G

tf.keras.utils.set_random_seed(42)
L, T, M = 10.0, 1.0, 100
xs = np.linspace(0.0, L, M, endpoint=False).astype(np.float32)   # sensor points
print("TensorFlow", tf.__version__)

## Experiment 1 — trunk / branch gradient split at $F = 0$

`gradient_split_at_F0` zeroes every branch output layer (forcing $\boldsymbol\beta = 0$,
hence $F \equiv 0$) and reports the two gradient norms separately.

**Predictions:** trunk norm exactly 0 in every case; mass residual exactly 0 in every
case; branch norm nonzero except for lake at rest.

This single table replaces Proposition 1's proof and is worth putting in the paper.

In [ ]:
def make_case(name, h0_fn, b_fn):
    return dict(name=name, h0=h0_fn, b=b_fn)

CASES = [
    make_case("C1  flat bed, h0 = 1+0.5 exp(-2(x-5)^2)",
              lambda x: 1.0 + 0.5 * tf.exp(-2.0 * (x - 5.0) ** 2),
              lambda x: tf.zeros_like(x)),
    make_case("C2  bump bathymetry",
              lambda x: 1.0 + 0.5 * tf.exp(-2.0 * (x - 5.0) ** 2),
              lambda x: 0.2 * tf.exp(-(x - 5.0) ** 2)),
    make_case("LAKE AT REST  h0 - b = 1.5 (exact steady state)",
              lambda x: 1.5 - 0.2 * tf.exp(-(x - 5.0) ** 2),
              lambda x: 0.2 * tf.exp(-(x - 5.0) ** 2)),
    make_case("FLAT WATER  h0 = 1, b = 0 (also a steady state)",
              lambda x: tf.ones_like(x),
              lambda x: tf.zeros_like(x)),
]

NC = 4000
rng = np.random.default_rng(0)
xc = tf.constant(rng.uniform(0, L, (NC, 1)).astype(np.float32))
tc = tf.constant(rng.uniform(0, T, (NC, 1)).astype(np.float32))
xs_t = tf.constant(xs[None, :])

model = FourBranchDeepONet(m=M, p=64, fusion="add", ic_mode="paper")
_ = model([tf.tile(xs_t, [NC, 1]) * 0 + 1.0, tf.tile(xs_t, [NC, 1]) * 0,
           tf.ones((NC, 1)), tf.zeros((NC, 1)), xc, tc])   # build

print(f"{'case':<48}{'||g_trunk||':>13}{'||g_branch||':>14}{'rms R1':>11}{'rms R2':>11}")
for c in CASES:
    h0s = tf.tile(c['h0'](xs_t), [NC, 1])
    bs  = tf.tile(c['b'](xs_t),  [NC, 1])
    r = gradient_split_at_F0(model, h0s, bs, c['h0'], c['b'], xc, tc)
    print(f"{c['name']:<48}{r['trunk_grad_norm']:>13.3e}{r['branch_grad_norm']:>14.3e}"
          f"{r['mass_residual_rms']:>11.3e}{r['momentum_residual_rms']:>11.3e}")

**How to read the table.** If `||g_trunk||` is 0 everywhere, `rms R1` is 0 everywhere,
`||g_branch||` is nonzero for C1/C2 and ~0 for the two steady states, then the corrected
statement holds and the original Proposition 1 does not. Report exactly this table.

## Experiment 2 — the surviving gradient points at hydrostatic balance

Decompose $L_{PDE}$ into its mass and momentum halves and confirm that at $F=0$ the
momentum residual equals the hydrostatic imbalance
$\partial_x(\tfrac12 g h_0^2) + g h_0 \partial_x b$ — i.e. the loss is measuring
*departure from lake at rest*, not departure from the true wave solution.

In [ ]:
def hydrostatic_imbalance(h0_fn, b_fn, x):
    with tf.GradientTape(persistent=True) as g:
        g.watch(x)
        h0, b = h0_fn(x), b_fn(x)
        press = 0.5 * G * h0 ** 2
    def _d(y, v):
        gr = g.gradient(y, v)
        return tf.zeros_like(v) if gr is None else gr
    r = _d(press, x) + G * h0 * _d(b, x)
    del g
    return r

for c in CASES:
    h0s = tf.tile(c['h0'](xs_t), [NC, 1])
    bs  = tf.tile(c['b'](xs_t),  [NC, 1])
    # force F = 0 by zeroing branch output layers
    saved = []
    for out in ("h", "hu"):
        for net in model.branch[out]:
            w, bv = net.layers[-1].get_weights()
            saved.append((net, w.copy(), bv.copy()))
            net.layers[-1].set_weights([np.zeros_like(w), np.zeros_like(bv)])
    r1, r2 = swe_residual(model, h0s, bs, c['h0'], c['b'], xc, tc)
    for net, w, bv in saved:
        net.layers[-1].set_weights([w, bv])
    hyd = hydrostatic_imbalance(c['h0'], c['b'], xc)
    print(f"{c['name'][:44]:<46} ||R2 - hydrostatic imbalance|| / ||R2|| = "
          f"{float(tf.norm(r2 - hyd) / (tf.norm(r2) + 1e-12)):.3e}")

## Experiment 3 — PI-DeepONet training: balanced vs unbalanced

The decisive dynamical test. Train the *purely physics-informed* variant on:

- **(a)** a lake-at-rest configuration → $F=0$ is an exact minimum, collapse should be total;
- **(b)** the C1/C2 configurations → gradient is nonzero, so the model should move, but
  toward hydrostatic balance rather than the wave solution.

If (b) shows the model drifting to a *steady state that is not $h_0$* — rather than sitting
exactly at $h_0$ as the paper reports — the reformulated claim is confirmed and the
"exact stationary point" language must go.

In [ ]:
def pde_loss(model, h0s, bs, h0_fn, b_fn, x, t):
    r1, r2 = swe_residual(model, h0s, bs, h0_fn, b_fn, x, t)
    return tf.reduce_mean(r1 ** 2) + tf.reduce_mean(r2 ** 2)

def train_pi(case, steps=3000, lr=1e-3, log_every=250):
    tf.keras.utils.set_random_seed(0)
    mdl = FourBranchDeepONet(m=M, p=64, fusion="add", ic_mode="exp")
    opt = tf.keras.optimizers.Adam(lr)
    h0s = tf.tile(case['h0'](xs_t), [NC, 1])
    bs  = tf.tile(case['b'](xs_t),  [NC, 1])
    hist = []
    for k in range(steps + 1):
        with tf.GradientTape() as tape:
            Lp = pde_loss(mdl, h0s, bs, case['h0'], case['b'], xc, tc)
        gs = tape.gradient(Lp, mdl.trainable_variables)
        gs, gn = tf.clip_by_global_norm(gs, 1.0)
        opt.apply_gradients(zip(gs, mdl.trainable_variables))
        if k % log_every == 0:
            beta_n = float(tf.norm(mdl.beta("h", h0s[:32], bs[:32]), axis=-1).numpy().mean())
            hist.append((k, float(Lp), float(gn), beta_n))
    return mdl, hist

for c in (CASES[2], CASES[0], CASES[1]):     # lake at rest first
    mdl, hist = train_pi(c, steps=3000)
    print(f"\n{c['name']}")
    print(f"  {'step':>6}{'L_PDE':>13}{'||grad||':>12}{'mean ||beta_h||':>18}")
    for k, lp, gn, bn in hist:
        print(f"  {k:>6}{lp:>13.4e}{gn:>12.4e}{bn:>18.4e}")

In [ ]:
# where did the unbalanced run actually end up?
xq = tf.constant(np.linspace(0, L, 500, dtype=np.float32)[:, None])
tq = tf.ones_like(xq) * T
c = CASES[1]
mdl, _ = train_pi(c, steps=3000)
h0s = tf.tile(c['h0'](xs_t), [500, 1]); bs = tf.tile(c['b'](xs_t), [500, 1])
h_pred, hu_pred = mdl([h0s, bs, c['h0'](xq), c['b'](xq), xq, tq])
h_pred = h_pred.numpy().ravel()

xn = xq.numpy().ravel()
h0n = c['h0'](xq).numpy().ravel(); bn_ = c['b'](xq).numpy().ravel()
lake = (h0n + bn_).mean() - bn_          # the lake-at-rest state with the same volume

plt.figure(figsize=(7, 4))
plt.plot(xn, h0n, 'k--', label=r'$h_0(x)$  (the "$F=0$" state)')
plt.plot(xn, lake, 'g-.', label='lake at rest, same volume')
plt.plot(xn, h_pred, 'r-', lw=2, label='PI-DeepONet at T=1 s')
plt.xlabel('x [m]'); plt.ylabel('h [m]'); plt.legend(); plt.title('Where does PI training go?')
plt.tight_layout(); plt.show()

print(f"||h_pred - h0||   = {np.linalg.norm(h_pred-h0n):.4f}")
print(f"||h_pred - lake|| = {np.linalg.norm(h_pred-lake):.4f}")
print("-> if the second is smaller, the attractor is the steady-state manifold, not h0.")

## Experiment 4 — re-measure the PDE gradient norm properly

§3.7.3 reports $\|\nabla_\theta L_{PDE}\| \approx 6.6\times10^{12}$, Remark 3 reports
$1.5\times10^2$ by finite differences, and the Fig. 6 caption says $2.2\times10^1$. Three
numbers, three places. A referee will read $10^{12}$ as a bug (division by a near-zero
$h$, or a norm over an unreduced loss).

Measure it per-network, at the paper's initialisation, with the ELU floor active.

In [ ]:
tf.keras.utils.set_random_seed(42)
mdl = FourBranchDeepONet(m=M, p=64, fusion="add", ic_mode="paper")
c = CASES[0]
h0s = tf.tile(c['h0'](xs_t), [NC, 1]); bs = tf.tile(c['b'](xs_t), [NC, 1])

with tf.GradientTape(persistent=True) as tape:
    Lp = pde_loss(mdl, h0s, bs, c['h0'], c['b'], xc, tc)
groups = {"branch_h": [v for n in mdl.branch['h'] for v in n.trainable_variables],
          "branch_hu": [v for n in mdl.branch['hu'] for v in n.trainable_variables],
          "trunk_h": mdl.trunk['h'].trainable_variables,
          "trunk_hu": mdl.trunk['hu'].trainable_variables}
print(f"L_PDE at init = {float(Lp):.4e}")
for name, vs in groups.items():
    g = [x for x in tape.gradient(Lp, vs) if x is not None]
    print(f"  ||grad|| {name:<11} = {float(tf.linalg.global_norm(g)):.4e}")
print(f"  ||grad|| TOTAL      = "
      f"{float(tf.linalg.global_norm([x for x in tape.gradient(Lp, mdl.trainable_variables) if x is not None])):.4e}")
del tape
print("\nReport ONE number, from automatic differentiation, and make Fig. 6's caption match.")

## Experiment 5 — fix the IC shortcut

Two defects in Eq. (12):

- **Not exact at $t=0$**: $\hat h(x,0) = h_0(x) + \epsilon$, because $\epsilon = 10^{-4}$ is
  added outside the ELU.
- **Positivity is not guaranteed**: $\mathrm{elu}(z) > -1$, so
  $\hat h > b + h_{\min} + \epsilon - 1$, which is **negative for $b < 0.95$ m**. The stated
  floor $\hat h \ge b + h_{\min} + \epsilon$ is false.

`ic_mode` offers three replacements. Verify all four numerically.

In [ ]:
xq = tf.constant(np.linspace(0, L, 400, dtype=np.float32)[:, None])
c = CASES[1]
h0q, bq = c['h0'](xq), c['b'](xq)

print(f"{'ic_mode':<10}{'max|h(x,0)-h0|':>18}{'min h over t':>16}{'guaranteed floor':>20}")
for mode in ("paper", "shifted", "exp", "softplus"):
    tf.keras.utils.set_random_seed(1)
    mdl = FourBranchDeepONet(m=M, p=64, fusion="add", ic_mode=mode)
    h0s = tf.tile(c['h0'](xs_t), [400, 1]); bs = tf.tile(c['b'](xs_t), [400, 1])
    h_ic, hu_ic = mdl([h0s, bs, h0q, bq, xq, tf.zeros_like(xq)])
    ic_err = float(tf.reduce_max(tf.abs(h_ic - h0q)))

    # adversarial stress test: drive the correction field strongly negative
    Fstress = tf.fill(tf.shape(xq), tf.constant(-50.0))
    h_stress = mdl.shortcut_h(h0q, bq, Fstress, tf.ones_like(xq))
    floor_ok = bool(tf.reduce_min(h_stress) >= float(tf.reduce_min(bq)) + 0.05 - 1e-5)
    print(f"{mode:<10}{ic_err:>18.3e}{float(tf.reduce_min(h_stress)):>16.4f}"
          f"{str(floor_ok):>20}")
print("\n'exp' and 'softplus' are exact at t=0 AND keep a hard floor at b + h_min.")

### Text changes this notebook supports

- Replace Proposition 1 with the four-clause statement at the top; the proof is three
  lines and is *correct*.
- Rewrite Remark 2: the hyperbolic/parabolic distinction is the identically-vanishing
  **mass** residual under $hu(x,0)=0$, not the chain-rule argument (which is
  equation-agnostic).
- Rewrite Remark 3 around one consistently measured gradient norm; fix Fig. 6's caption.
- Rename "$F=0$ attractor" → "steady-state (lake-at-rest) attractor" in the title,
  abstract, keywords and §3.5.1.
- Add the Rohrhofer et al. (2023) and De Ryck et al. citations and soften
  "not previously characterised".
- Replace Eq. (12) with the `exp` or `softplus` form and drop the softplus-underflow
  paragraph (with the floor in place, $u = hu/h$ can never see a near-zero denominator).